# COMM117 — Week 2 Exercise: Tokenisation 



**Learning objectives**
- Understand why tokenisation is required (text → tokens → IDs)
- Implement simple word and character tokenisers
- Explore subword tokenisation ideas (BPE) at a toy scale
- Reflect on trade-offs: vocabulary size vs sequence length


## 0. Setup
Run the cell below to import libraries used in this exercise.

In [ ]:
import re
from collections import Counter
from typing import List, Tuple, Dict

print("Ready.")

## 1. Warm-up: what is a “token”?

A **tokeniser** maps raw text → a sequence of discrete symbols (tokens), then maps tokens → integer IDs.

In this section you will:
1. Split text into **word tokens** (simple baseline)
2. Split text into **word tokens with punctuation separated**
3. Count tokens and compare outputs


In [ ]:
text = "Tokenization is important for LLMs!"
print(text)

In [ ]:
# 1.1 TODO: naive word tokenisation (split on whitespace)

def word_tokenize(text: str) -> List[str]:
    # TODO: implement
   
    raise NotImplementedError

tokens = word_tokenize(text)
print(tokens)
print("Token count:", len(tokens))

In [ ]:
# 1.2 TODO: word tokenisation with punctuation split
# Requirement: keep punctuation marks as separate tokens.
# Example:
# "LLMs!" -> ["LLMs", "!"]

_word_pat = re.compile(r"[A-Za-z0-9]+|[^\w\s]")  # words OR single non-space punctuation

def word_tokenize_punct(text: str) -> List[str]:
    # TODO: implement using _word_pat
    
    raise NotImplementedError

tokens2 = word_tokenize_punct(text)
print(tokens2)
print("Token count:", len(tokens2))

## 2. Vocabulary & token IDs

A vocabulary is a mapping: **token → integer ID**.

In this section you will:
- Build a vocabulary from tokenised text
- Encode tokens into IDs and decode back


In [ ]:
corpus = [
    "I am a machine learning researcher.",
    "I work on large language models!",
    "Tokenization matters."
]
print("Corpus size:", len(corpus))

In [ ]:
# 2.1 TODO: build a vocabulary from a list of token lists
# Requirement: reserve ID 0 for a special token "<PAD>"

def build_vocab(token_lists: List[List[str]]) -> Dict[str, int]:
    # TODO: implement
    raise NotImplementedError

token_lists = [word_tokenize_punct(s.lower()) for s in corpus]
vocab = build_vocab(token_lists)

print("Vocab size:", len(vocab))
print("First 20 items:", list(vocab.items())[:20])

In [ ]:
# 2.2 TODO: encode and decode

def encode(tokens: List[str], vocab: Dict[str,int]) -> List[int]:
    # TODO: implement
    # Hint: vocab[token]
    raise NotImplementedError

def decode(ids: List[int], inv_vocab: Dict[int,str]) -> List[str]:
    # TODO: implement
    raise NotImplementedError

inv_vocab = {i:t for t,i in vocab.items()}

example = word_tokenize_punct("I love LLMs!".lower())
ids = encode(example, vocab)
back = decode(ids, inv_vocab)

print("Tokens:", example)
print("IDs:   ", ids)
print("Back:  ", back)

## 3. Character tokenisation 

Character tokenisation splits text into individual characters.

In this section you will:
- Implement a character tokeniser
- Compare token counts vs word/subword style


In [ ]:
# 3.1 TODO: implement a character tokeniser

def char_tokenize(text: str, keep_whitespace: bool = True) -> List[str]:
    # TODO: implement
    # If keep_whitespace is True, include spaces/newlines as tokens.
    # If False, drop them.
    raise NotImplementedError

ex = "LLMs!"
print(char_tokenize(ex, keep_whitespace=True))
print(char_tokenize(ex, keep_whitespace=False))

In [ ]:
# 3.2 Compare token counts across methods for a longer sentence

s = "The quick brown fox jumps over the lazy dog."

w = word_tokenize(s)
wp = word_tokenize_punct(s)
ch = char_tokenize(s, keep_whitespace=False)

print("Sentence:", s)
print("Word tokens:", len(w))
print("Word+punct tokens:", len(wp))
print("Char tokens (no whitespace):", len(ch))

### 3.3 Reflection

Why might character tokenisation increase compute cost for Transformers?


In [ ]:
reflection_3 = """TODO"""
print(reflection_3)

## 4. Toy Byte Pair Encoding (BPE) — subword idea

This is a **toy** implementation to understand the idea.
Real tokenisers (GPT-2 BPE, WordPiece, SentencePiece) are more complex.

We will:
1. Represent each word as characters separated by spaces (e.g., "l o w </w>")
2. Count adjacent symbol pairs
3. Merge the most frequent pair
4. Repeat for a small number of merges


In [ ]:
# Toy vocab from classic examples
toy_vocab = {
    "l o w </w>": 5,
    "l o w e r </w>": 2,
    "n e w e s t </w>": 6,
    "w i d e s t </w>": 3,
}
toy_vocab

In [ ]:


from collections import Counter

def get_pairs(vocab: Dict[str,int]) -> Counter:
    pairs = Counter()
    for word, freq in vocab.items():
        #TODO count adjacent pairs
    return pairs

pairs = get_pairs(toy_vocab)
pairs.most_common(10)[:10]

In [ ]:
# 4.2 TODO: merge a chosen pair in the vocab
# Requirement:
# - Replace occurrences of "A B" with "AB"
# - Preserve frequencies

def merge_vocab(vocab: Dict[str,int], pair: Tuple[str,str]) -> Dict[str,int]:
    # TODO: implement
    raise NotImplementedError

# Test: merge the most common pair once
most_common_pair = get_pairs(toy_vocab).most_common(1)[0][0]
print("Most common pair:", most_common_pair)

toy_vocab_1 = merge_vocab(toy_vocab, most_common_pair)
toy_vocab_1

In [ ]:
# 4.3 TODO: run a few merges and print what happened

def run_bpe(vocab: Dict[str,int], num_merges: int = 5):
    v = dict(vocab)
    merges = []
    for _ in range(num_merges):
        pairs = get_pairs(v)
        if not pairs:
            break
        pair = pairs.most_common(1)[0][0]
        v = merge_vocab(v, pair)
        merges.append(pair)
        print(f"Merged {pair} -> {''.join(pair)}")
    return v, merges

final_vocab, merges = run_bpe(toy_vocab, num_merges=5)
print("\nFinal vocab entries:")
for k,v in final_vocab.items():
    print(v, ":", k)